# Exploratory Data Analysis — SQuAD v1.1

This notebook performs comprehensive EDA on the SQuAD v1.1 dataset:
- Dataset structure and statistics
- Context / question / answer length distributions
- Token length analysis with max_seq_length planning
- Answer type analysis
- Sample inspection

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from datasets import load_dataset
from transformers import AutoTokenizer

sns.set_theme(style='whitegrid')
%matplotlib inline
print('Imports complete')

## 1. Load Dataset

In [ ]:
dataset = load_dataset('squad', cache_dir='../data/processed')
print(dataset)
train_df = dataset['train'].to_pandas()
val_df   = dataset['validation'].to_pandas()
print(f'Train: {len(train_df):,} | Val: {len(val_df):,}')

## 2. Basic Statistics

In [ ]:
for split, df in [('Train', train_df), ('Validation', val_df)]:
    ctx_len   = df['context'].str.split().str.len()
    q_len     = df['question'].str.split().str.len()
    ans_text  = df['answers'].apply(lambda x: x['text'][0] if x['text'] else '')
    ans_len   = ans_text.str.split().str.len()
    print(f'\n=== {split} ===')
    print(f'  Examples          : {len(df):,}')
    print(f'  Unique contexts   : {df["context"].nunique():,}')
    print(f'  Avg context words : {ctx_len.mean():.1f} (std={ctx_len.std():.1f})')
    print(f'  Max context words : {ctx_len.max()}')
    print(f'  Avg question words: {q_len.mean():.1f} (std={q_len.std():.1f})')
    print(f'  Avg answer words  : {ans_len.mean():.1f}')

## 3. Length Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ctx_len = train_df['context'].str.split().str.len()
axes[0].hist(ctx_len, bins=50, color=sns.color_palette('colorblind')[0], edgecolor='white')
axes[0].set_title('Context Length (words)', fontweight='bold')
axes[0].set_xlabel('Word count')
axes[0].set_ylabel('Frequency')

q_len = train_df['question'].str.split().str.len()
axes[1].hist(q_len, bins=30, color=sns.color_palette('colorblind')[1], edgecolor='white')
axes[1].set_title('Question Length (words)', fontweight='bold')
axes[1].set_xlabel('Word count')

ans_len = train_df['answers'].apply(lambda x: len(x['text'][0].split()) if x['text'] else 0)
axes[2].hist(ans_len, bins=30, color=sns.color_palette('colorblind')[2], edgecolor='white')
axes[2].set_title('Answer Length (words)', fontweight='bold')
axes[2].set_xlabel('Word count')

plt.tight_layout()
plt.savefig('../results/plots/eda_length_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Token Length Analysis (with Tokenizer)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased', use_fast=True)

# Sample 2000 examples for speed
sample = train_df.sample(2000, random_state=42)
token_lengths = [
    len(tokenizer(row['question'], row['context'])['input_ids'])
    for _, row in sample.iterrows()
]

print('Token length percentiles:')
for p in [25, 50, 75, 90, 95, 99]:
    print(f'  P{p:2d}: {np.percentile(token_lengths, p):.0f} tokens')

print(f'\nExamples exceeding 384 tokens: {sum(l > 384 for l in token_lengths) / len(token_lengths):.1%}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(token_lengths, bins=40, color=sns.color_palette('colorblind')[3], edgecolor='white')
ax.axvline(384, color='red', linestyle='--', linewidth=2, label='max_seq_length=384')
ax.axvline(256, color='orange', linestyle='--', linewidth=2, label='max_seq_length=256')
ax.set_xlabel('Total Token Length (Q + Context)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Input Token Length Distribution (DistilBERT tokenizer)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('../results/plots/token_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Answer Type Analysis

In [ ]:
def classify_answer(text):
    """Heuristic answer type classifier."""
    if not text:
        return 'empty'
    if re.match(r'^\d{4}$', text):
        return 'year'
    if re.match(r'^\d+(\.\d+)?\s*(percent|%|million|billion|km|miles)?$', text, re.I):
        return 'number'
    if re.match(r'^[A-Z][a-z]+(\s+[A-Z][a-z]+)*$', text):
        return 'proper noun'
    if len(text.split()) <= 2:
        return 'short phrase'
    return 'long phrase'

answer_types = train_df['answers'].apply(
    lambda x: classify_answer(x['text'][0] if x['text'] else '')
)
type_counts = answer_types.value_counts()

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(type_counts.index, type_counts.values,
        color=sns.color_palette('colorblind'), edgecolor='white')
for i, v in enumerate(type_counts.values):
    ax.text(v + 100, i, f'{v:,} ({v/len(train_df):.1%})', va='center', fontsize=10)
ax.set_xlabel('Count', fontsize=12)
ax.set_title('Answer Type Distribution (SQuAD Train)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/plots/answer_type_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Question Type Analysis

In [ ]:
def get_question_type(q):
    q_lower = q.lower()
    for wh in ['what', 'who', 'when', 'where', 'why', 'how', 'which', 'whom']:
        if q_lower.startswith(wh):
            return wh.capitalize()
    return 'Other'

q_types = train_df['question'].apply(get_question_type)
q_type_counts = q_types.value_counts()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(q_type_counts.index, q_type_counts.values,
              color=sns.color_palette('colorblind'), edgecolor='white')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{bar.get_height():,}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Question Type Distribution (SQuAD Train)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/plots/question_type_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('Question type distribution:')
for qt, cnt in q_type_counts.items():
    print(f'  {qt:10s}: {cnt:6,} ({cnt/len(train_df):.1%})')

## 7. Sample Examples

In [ ]:
samples = train_df.sample(3, random_state=7)
for _, row in samples.iterrows():
    print('─' * 70)
    print(f"ID       : {row['id']}")
    print(f"Context  : {row['context'][:300]}...")
    print(f"Question : {row['question']}")
    print(f"Answer   : {row['answers']['text'][0]}")
    print(f"Char pos : {row['answers']['answer_start'][0]}")
print('─' * 70)